In [1]:
import xarray as xr
import pandas as pd
import numpy as np
nc = xr.open_dataset("../../NC/compare_diff.nc")

basin_diff = nc["basin_agri_diff"]
region_diff = nc["region_agri_diff"]

In [2]:
def excess_kurtosis_xarray(da):
    """
    da: xarray.DataArray, 只包含空间维度（单一年份）
    """
    values = da.values.flatten()
    values = values[~np.isnan(values)]

    if values.size == 0:
        return np.nan

    mu = np.mean(values)
    sigma = np.std(values, ddof=0)  # 用 1/n，与公式一致

    if sigma == 0:
        return np.nan

    kurt = np.mean(((values - mu) / sigma) ** 4) - 3
    return kurt

In [3]:
years = range(2010, 2101,10)

kurt_basin  = []
kurt_region = []

for y in years:
    time = f"{y}-01-01"
    basin_y  = basin_diff.sel(time=time)
    region_y = region_diff.sel(time=time)

    kurt_basin.append(excess_kurtosis_xarray(basin_y))
    kurt_region.append(excess_kurtosis_xarray(region_y))

In [4]:
import pandas as pd
df = pd.DataFrame({
    "year": years,
    "kurt_basin": kurt_basin,
    "kurt_region": kurt_region
})
df.to_csv("../../CSV/interannual_diff_kurtosis.csv", index=False)